<a href="https://colab.research.google.com/github/tuanhoangcg2007-pixel/HKT_KTHP_BAINOP/blob/main/M%C3%B4hinhtrain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
PATH_DATASET = '/content/drive/MyDrive/KTHP'
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)
valid_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

print("🚀 Đang trích xuất dữ liệu để huấn luyện (Train):")
train_generator = train_datagen.flow_from_directory(
    PATH_DATASET,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='history' if hasattr(train_datagen, 'subset') else 'training' # 🌟 Lấy 80% cho tập Train
)

print("\n📸 Đang trích xuất dữ liệu để tự đánh giá (Validation):")
validation_generator = valid_datagen.flow_from_directory(
    PATH_DATASET,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

print("\n✅ Danh sách món ăn đã nhận diện thành công:")
print(train_generator.class_indices)

🚀 Đang trích xuất dữ liệu để huấn luyện (Train):
Found 7234 images belonging to 11 classes.

📸 Đang trích xuất dữ liệu để tự đánh giá (Validation):
Found 1806 images belonging to 11 classes.

✅ Danh sách món ăn đã nhận diện thành công:
{'Canh chua': 0, 'Canh chua cá': 1, 'Canh rau': 2, 'Cá hú kho': 3, 'Cơm trắng': 4, 'Rau củ xào': 5, 'Sườn nướng': 6, 'Thịt kho': 7, 'Thịt kho trứng': 8, 'Trứng chiên': 9, 'Đậu hủ sốt cà': 10}


In [ ]:
import os
import math
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ==========================================================
# THÔNG SỐ CẤU HÌNH (BẠN CÓ THỂ ĐIỀU CHỈNH TẠI ĐÂY)
# ==========================================================
PATH_DATASET = '/content/drive/MyDrive/KTHP'
NEW_CHECKPOINT_PATH = '/content/drive/MyDrive/MoHinh/mo_hinh_mobilenet_12_mon_HOAN_HAO.h5'
BATCH_SIZE = 32
EPOCHS = 30

# Tự động tạo thư mục lưu mô hình nếu chưa có
os.makedirs(os.path.dirname(NEW_CHECKPOINT_PATH), exist_ok=True)

# ==========================================================
# PHASE 1: ĐỌC VÀ CHIA DỮ LIỆU TỰ ĐỘNG (80% TRAIN / 20% VALIDATION)
# ==========================================================
# Cấu hình Augmentation cho tập Train để chống học vẹt (Overfitting)
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2  # Giữ lại 20% dữ liệu để validation
)

# Tập Validation CHỈ chuẩn hóa ảnh thô, không bóp méo ảnh để đánh giá khách quan
valid_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2  # Đồng bộ tỷ lệ chia 20%
)

print("🚀 1. Đang trích xuất 80% dữ liệu để huấn luyện (Train)...")
train_generator = train_datagen.flow_from_directory(
    PATH_DATASET,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'     # Lấy 80% cho tập Train
)

print("\n📸 2. Đang trích xuất 20% dữ liệu để tự đánh giá (Validation)...")
validation_generator = valid_datagen.flow_from_directory(
    PATH_DATASET,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'    # Lấy 20% còn lại cho tập Validation
)

print("\n✅ Danh sách 12 món ăn đã cấu hình thành công:")
print(train_generator.class_indices)
print("-" * 50)

# ==========================================================
# PHASE 2: KHỞI TẠO KIẾN TRÚC MÔ HÌNH (TỰ ĐỘNG ĐẾM SỐ MÓN ĂN)
# ==========================================================
# 🌟 ĐOẠN FIX LỖI: Tự động lấy số lượng class thực tế từ train_generator
NUM_CLASSES = len(train_generator.class_indices)

print(f"🧠 3. Đang khởi tạo mô hình mạng MobileNetV2 cho {NUM_CLASSES} món ăn...")
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Đóng băng các tầng cốt lõi của MobileNetV2
for layer in base_model.layers:
    layer.trainable = False

# Xây dựng các tầng phân loại tùy biến (Top Layers)
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)

# Đầu ra tự động khớp với số lượng thư mục thực tế tìm được
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

model_transfer = Model(inputs=base_model.input, outputs=predictions)
model_transfer.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ==========================================================
# PHASE 3: CẤU HÌNH PHƯƠNG THỨC GIÁM SÁT (CALLBACKS)
# ==========================================================
# Tự động lưu lại phiên bản có val_accuracy cao kỷ lục
checkpoint = ModelCheckpoint(
    filepath=NEW_CHECKPOINT_PATH,
    monitor='val_accuracy',
    mode='max',
    save_best_only=True,
    verbose=1
)

# Nếu sau 5 vòng (epochs) mà val_accuracy không cải thiện thì dừng để tránh mất thời gian & học vẹt
early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# ==========================================================
# PHASE 4: TÍNH TOÁN BƯỚC CHẠY VÀ KÍCH HOẠT HUẤN LUYỆN
# ==========================================================
compute_steps_per_epoch = math.ceil(train_generator.samples / train_generator.batch_size)
compute_validation_steps = math.ceil(validation_generator.samples / validation_generator.batch_size)

print("\n📊 THÔNG SỐ ĐƯỜNG CHẠY:")
print(f"   - Tổng số ảnh tập Train: {train_generator.samples} ảnh ({compute_steps_per_epoch} bước/vòng)")
print(f"   - Tổng số ảnh tập Validation: {validation_generator.samples} ảnh ({compute_validation_steps} bước/vòng)")
print("\n🔥 BẮT ĐẦU CHẠY HUẤN LUYỆN...")

# Tiến hành chạy huấn luyện chính thức
history = model_transfer.fit(
    train_generator,
    epochs=EPOCHS,
    steps_per_epoch=compute_steps_per_epoch,
    validation_data=validation_generator,
    validation_steps=compute_validation_steps,
    callbacks=[checkpoint, early_stopping],
    verbose=1
)

print(f"\n🎉 QUÁ TRÌNH HUẤN LUYỆN HOÀN TẤT! File mô hình chuẩn đét đã được lưu tại: {NEW_CHECKPOINT_PATH}")

🚀 1. Đang trích xuất 80% dữ liệu để huấn luyện (Train)...
Found 7234 images belonging to 11 classes.

📸 2. Đang trích xuất 20% dữ liệu để tự đánh giá (Validation)...
Found 1806 images belonging to 11 classes.

✅ Danh sách 12 món ăn đã cấu hình thành công:
{'Canh chua': 0, 'Canh chua cá': 1, 'Canh rau': 2, 'Cá hú kho': 3, 'Cơm trắng': 4, 'Rau củ xào': 5, 'Sườn nướng': 6, 'Thịt kho': 7, 'Thịt kho trứng': 8, 'Trứng chiên': 9, 'Đậu hủ sốt cà': 10}
--------------------------------------------------
🧠 3. Đang khởi tạo mô hình mạng MobileNetV2 cho 11 món ăn...

📊 THÔNG SỐ ĐƯỜNG CHẠY:
   - Tổng số ảnh tập Train: 7234 ảnh (227 bước/vòng)
   - Tổng số ảnh tập Validation: 1806 ảnh (57 bước/vòng)

🔥 BẮT ĐẦU CHẠY HUẤN LUYỆN...
Epoch 1/30
227/227 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.6521 - loss: 1.0690
Epoch 1: val_accuracy improved from None to 0.84718, saving model to /content/drive/MyDrive/MoHinh/mo_hinh_mobilenet_12_mon_HOAN_HAO.h5



Epoch 1: finished saving model to /content/drive/MyDrive/MoHinh/mo_hinh_mobilenet_12_mon_HOAN_HAO.h5
227/227 ━━━━━━━━━━━━━━━━━━━━ 2520s 11s/step - accuracy: 0.7795 - loss: 0.6443 - val_accuracy: 0.8472 - val_loss: 0.4518
Epoch 2/30
227/227 ━━━━━━━━━━━━━━━━━━━━ 0s 651ms/step - accuracy: 0.8997 - loss: 0.3099
Epoch 2: val_accuracy improved from 0.84718 to 0.84828, saving model to /content/drive/MyDrive/MoHinh/mo_hinh_mobilenet_12_mon_HOAN_HAO.h5



Epoch 2: finished saving model to /content/drive/MyDrive/MoHinh/mo_hinh_mobilenet_12_mon_HOAN_HAO.h5
227/227 ━━━━━━━━━━━━━━━━━━━━ 163s 720ms/step - accuracy: 0.9032 - loss: 0.2970 - val_accuracy: 0.8483 - val_loss: 0.4592
Epoch 3/30
227/227 ━━━━━━━━━━━━━━━━━━━━ 0s 653ms/step - accuracy: 0.9236 - loss: 0.2180
Epoch 3: val_accuracy improved from 0.84828 to 0.85382, saving model to /content/drive/MyDrive/MoHinh/mo_hinh_mobilenet_12_mon_HOAN_HAO.h5



Epoch 3: finished saving model to /content/drive/MyDrive/MoHinh/mo_hinh_mobilenet_12_mon_HOAN_HAO.h5
227/227 ━━━━━━━━━━━━━━━━━━━━ 202s 722ms/step - accuracy: 0.9259 - loss: 0.2161 - val_accuracy: 0.8538 - val_loss: 0.4351
Epoch 4/30
227/227 ━━━━━━━━━━━━━━━━━━━━ 0s 653ms/step - accuracy: 0.9431 - loss: 0.1726
Epoch 4: val_accuracy improved from 0.85382 to 0.88206, saving model to /content/drive/MyDrive/MoHinh/mo_hinh_mobilenet_12_mon_HOAN_HAO.h5



Epoch 4: finished saving model to /content/drive/MyDrive/MoHinh/mo_hinh_mobilenet_12_mon_HOAN_HAO.h5
227/227 ━━━━━━━━━━━━━━━━━━━━ 164s 721ms/step - accuracy: 0.9399 - loss: 0.1799 - val_accuracy: 0.8821 - val_loss: 0.4022
Epoch 5/30
227/227 ━━━━━━━━━━━━━━━━━━━━ 0s 644ms/step - accuracy: 0.9426 - loss: 0.1645
Epoch 5: val_accuracy improved from 0.88206 to 0.88483, saving model to /content/drive/MyDrive/MoHinh/mo_hinh_mobilenet_12_mon_HOAN_HAO.h5



Epoch 5: finished saving model to /content/drive/MyDrive/MoHinh/mo_hinh_mobilenet_12_mon_HOAN_HAO.h5
227/227 ━━━━━━━━━━━━━━━━━━━━ 162s 711ms/step - accuracy: 0.9414 - loss: 0.1662 - val_accuracy: 0.8848 - val_loss: 0.3749
Epoch 6/30
227/227 ━━━━━━━━━━━━━━━━━━━━ 0s 648ms/step - accuracy: 0.9525 - loss: 0.1383
Epoch 6: val_accuracy did not improve from 0.88483
227/227 ━━━━━━━━━━━━━━━━━━━━ 162s 711ms/step - accuracy: 0.9530 - loss: 0.1397 - val_accuracy: 0.8682 - val_loss: 0.4530
Epoch 7/30
227/227 ━━━━━━━━━━━━━━━━━━━━ 0s 649ms/step - accuracy: 0.9502 - loss: 0.1450
Epoch 7: val_accuracy did not improve from 0.88483
227/227 ━━━━━━━━━━━━━━━━━━━━ 162s 713ms/step - accuracy: 0.9516 - loss: 0.1347 - val_accuracy: 0.8738 - val_loss: 0.5020
Epoch 8/30
227/227 ━━━━━━━━━━━━━━━━━━━━ 0s 635ms/step - accuracy: 0.9590 - loss: 0.1230
Epoch 8: val_accuracy did not improve from 0.88483
227/227 ━━━━━━━━━━━━━━━━━━━━ 158s 697ms/step - accuracy: 0.9594 - loss: 0.1192 - val_accuracy: 0.8605 - val_loss: 0.53